# Black-Box Optimisation Capstone

This notebook presents the portable core of the project: loading the eight course datasets, fitting a Gaussian Process surrogate, ranking candidates with Upper Confidence Bound, validating portal queries and summarising the final results. The full week-by-week decision record is linked from the repository README.

## 1. Setup and reproducibility

Install dependencies with `pip install -r requirements.txt`. The course-provided arrays are intentionally not redistributed; place them under `data/initial_data/` as described in `data/README.md`. All random operations below use explicit seeds.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import qmc
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'initial_data'
RANDOM_SEED = 2026
print(f'Project root: {PROJECT_ROOT}')
print(f'Initial data available: {DATA_DIR.exists()}')

In [ ]:
def load_initial_data(data_dir: Path) -> dict[int, tuple[np.ndarray, np.ndarray]]:
    datasets = {}
    for function_id in range(1, 9):
        folder = data_dir / f'function_{function_id}'
        x_path = folder / 'initial_inputs.npy'
        y_path = folder / 'initial_outputs.npy'
        if not x_path.exists() or not y_path.exists():
            raise FileNotFoundError(
                f'Missing Function {function_id} arrays. Follow data/README.md.'
            )
        X = np.load(x_path).astype(float)
        y = np.load(y_path).astype(float).reshape(-1)
        if X.ndim != 2 or len(X) != len(y):
            raise ValueError(f'Invalid shapes for Function {function_id}: {X.shape}, {y.shape}')
        if not np.all((X >= 0) & (X <= 1)):
            raise ValueError(f'Function {function_id} contains out-of-bounds inputs')
        datasets[function_id] = (X, y)
    return datasets

datasets = load_initial_data(DATA_DIR) if DATA_DIR.exists() else {}
if datasets:
    pd.DataFrame([
        {'function': i, 'observations': len(y), 'dimensions': X.shape[1], 'initial_best': y.max()}
        for i, (X, y) in datasets.items()
    ])

## 2. Gaussian Process and UCB baseline

The early-round baseline fitted one Gaussian Process per function. UCB scores each candidate as `predicted mean + kappa × predicted standard deviation`. Larger `kappa` increases exploration. Later rounds supplemented this baseline with Expected Improvement, local regressions, RBF SVMs, neural-network sensitivity checks and transparent local heuristics.

In [ ]:
def fit_gp(X: np.ndarray, y: np.ndarray, seed: int = RANDOM_SEED) -> GaussianProcessRegressor:
    dimensions = X.shape[1]
    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(length_scale=np.full(dimensions, 0.25), length_scale_bounds=(0.01, 10), nu=2.5)
        + WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-7, 0.8))
    )
    return GaussianProcessRegressor(
        kernel=kernel, normalize_y=True, n_restarts_optimizer=5, random_state=seed
    ).fit(X, y)

def sobol_candidates(dimensions: int, power: int = 14, seed: int = RANDOM_SEED) -> np.ndarray:
    return qmc.Sobol(dimensions, scramble=True, seed=seed).random_base2(power)

def rank_ucb(model, candidates: np.ndarray, kappa: float = 1.0) -> pd.DataFrame:
    mean, std = model.predict(candidates, return_std=True)
    table = pd.DataFrame({'mean': mean, 'std': std, 'ucb': mean + kappa * std})
    return table.sort_values('ucb', ascending=False)

# Example after the course data are installed:
if datasets:
    X_example, y_example = datasets[1]
    gp = fit_gp(X_example, np.log10(np.maximum(y_example, 1e-130)))
    candidates = sobol_candidates(X_example.shape[1])
    display(rank_ucb(gp, candidates, kappa=1.0).head())

## 3. Query validation

Every portal query had to match the correct dimensionality, stay within bounds, use six decimal places and avoid previously evaluated points.

In [ ]:
def validate_query(x, dimensions: int, existing=None, tolerance: float = 5e-7) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if x.shape != (dimensions,):
        raise ValueError(f'Expected {dimensions} coordinates; received {x.shape}')
    if np.any((x < 0) | (x > 0.999999)):
        raise ValueError('Coordinates must lie between 0.000000 and 0.999999')
    rounded = np.round(x, 6)
    if existing is not None and np.any(np.linalg.norm(np.asarray(existing) - rounded, axis=1) <= tolerance):
        raise ValueError('Query duplicates an existing observation')
    return rounded

def portal_format(x) -> str:
    return '-'.join(f'{value:.6f}' for value in x)

portal_format(validate_query([0.579980, 0.579580], dimensions=2))

## 4. Final results

All objectives were maximised. Negative values for Functions 3 and 6 therefore improve as they move closer to zero. Cross-function magnitudes are not directly comparable.

In [ ]:
final_results = pd.DataFrame([
    (1, 7.710875114502849e-16, 0.0001362433711915106, 13),
    (2, 0.6112052157614438, 0.6859447566834628, 9),
    (3, -0.034835313350078584, -0.0028315397301727427, 6),
    (4, -4.025542281908162, 0.6789254464625576, 13),
    (5, 1088.8596181962705, 1636.5047451170747, 13),
    (6, -0.7142649478202404, -0.3056501399168364, 3),
    (7, 1.3649683044991994, 1.8599092342834425, 13),
    (8, 9.598482002566342, 9.9429688307286, 12),
], columns=['function', 'best_initial', 'best_final', 'first_best_week'])
final_results['absolute_improvement'] = final_results.best_final - final_results.best_initial
final_results

In [ ]:
plot_data = final_results.set_index('function')['first_best_week']
ax = plot_data.plot.bar(figsize=(9, 4), color='#24356b', title='Round in which the final best result was first achieved')
ax.set_xlabel('Function')
ax.set_ylabel('Week')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Interpretation

The final best output exceeded the initial incumbent for every function. Controlled directional continuation was especially effective for Functions 4 and 7. Function 5 became a boundary-constrained, effectively one-dimensional search. Functions 2 and 6 remained locally unstable, and Function 8 plateaued. These are empirical improvements, not proofs of global optimality; the adaptive history is concentrated around promising regions and the true functions remain unknown. See `docs/MODEL_CARD.md` and `docs/DATASHEET.md` for detailed limitations and responsible-use guidance.